# Sentiment Analysis – IMDB Movie Reviews

## Imports

Here I'm loading all the tools I need: pandas for the data, matplotlib and seaborn for the charts, and scikit-learn to build the model.

In [ ]:
import re
import string

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_style("whitegrid")

## Load & explore the data

I loaded the IMDB dataset from Kaggle. It has 50,000 movie reviews, and each one is labeled as positive or negative.

In [ ]:
df = pd.read_csv("IMDB Dataset.csv")
print("Shape:", df.shape)
df.head()

### Checking the data
I checked for empty values and repeated reviews. There were no empty values, but I found some duplicates, so I removed them so the model doesn't see the same review twice.

In [ ]:
df.info()
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
# Remove duplicate reviews
df = df.drop_duplicates().reset_index(drop=True)
print("Shape after removing duplicates:", df.shape)

### Positive vs negative
I wanted to see if the data is balanced. It is: there are almost the same number of positive and negative reviews, which is good because the model won't favor one side.

In [ ]:
# Class balance
ax = sns.countplot(x="sentiment", data=df, hue="sentiment", palette=["#d9534f", "#5cb85c"], legend=False)
ax.set_title("Number of reviews per sentiment")
plt.show()
df["sentiment"].value_counts()

### How long are the reviews?
Most reviews are between 100 and 300 words. Positive and negative reviews have about the same length, so length alone can't tell us the sentiment.

In [ ]:
# Review length (in words) per sentiment
df["word_count"] = df["review"].str.split().str.len()
sns.histplot(data=df, x="word_count", hue="sentiment", bins=60, element="step")
plt.xlim(0, 1000)
plt.title("Review length distribution")
plt.show()
df.groupby("sentiment")["word_count"].describe()

## Clean the text

The reviews had HTML tags, punctuation and capital letters, so I cleaned them up. I kept words like "not" and "never" because they change the meaning ("not good" is not "good").

In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"<.*?>", " ", text)                 # HTML tags
    text = re.sub(r"http\S+|www\.\S+", " ", text)     # URLs
    text = re.sub(r"[^a-z\s]", " ", text)              # punctuation & numbers
    text = re.sub(r"\s+", " ", text).strip()           # extra spaces
    return text

print("BEFORE:", df["review"][1][:300])
print("\nAFTER: ", clean_text(df["review"][1])[:300])

In [ ]:
df["clean_review"] = df["review"].apply(clean_text)
df["label"] = df["sentiment"].map({"negative": 0, "positive": 1})
df[["clean_review", "sentiment", "label"]].head()

### Most frequent words per class

Positive reviews use words like "great", "love" and "best". Negative reviews use words like "bad", "worst" and "boring". This shows the words really do carry the feeling.

In [ ]:
NEGATIONS = {"no", "not", "nor", "never", "none", "nothing", "nobody", "neither", "cannot"}
STOP_WORDS = list(ENGLISH_STOP_WORDS - NEGATIONS) + ["br", "movie", "film", "one"]

def top_words(texts, n=15):
    words = " ".join(texts).split()
    words = [w for w in words if w not in STOP_WORDS and len(w) > 2]
    return pd.Series(words).value_counts().head(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (label, color) in zip(axes, [("positive", "#5cb85c"), ("negative", "#d9534f")]):
    tw = top_words(df.loc[df["sentiment"] == label, "clean_review"].sample(5000, random_state=42))
    tw.sort_values().plot.barh(ax=ax, color=color)
    ax.set_title(f"Top words – {label} reviews")
plt.tight_layout()
plt.show()

## Train_test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_review"], df["label"],
    test_size=0.2, random_state=42, stratify=df["label"]
)
print("Train:", len(X_train), "| Test:", len(X_test))

I split the data: 80% to train the model and 20% to test it. This way I test the model on reviews it has never seen before.

## TF-IDF + compare models

Computers don't understand words, so I used TF-IDF to turn each review into numbers. Then I trained three models and compared their accuracy. Logistic Regression and Linear SVM did the best.

In [ ]:
def make_pipeline(model):
    return Pipeline([
        ("tfidf", TfidfVectorizer(stop_words=STOP_WORDS, ngram_range=(1, 2),
                                  max_features=50000, min_df=2, sublinear_tf=True)),
        ("clf", model),
    ])

models = {
    "Naive Bayes":          MultinomialNB(),
    "Logistic Regression":  LogisticRegression(max_iter=1000, C=2.0),
    "Linear SVM":           LinearSVC(C=0.5),
}

results = {}
for name, model in models.items():
    pipe = make_pipeline(model)
    pipe.fit(X_train, y_train)
    acc = accuracy_score(y_test, pipe.predict(X_test))
    results[name] = acc
    print(f"{name:<22} accuracy: {acc:.4f}")

In [ ]:
pd.Series(results).sort_values().plot.barh(color="#337ab7", figsize=(7, 3))
plt.xlim(0.8, 0.95)
plt.title("Test accuracy by model")
plt.show()

## Evaluate the final model

### Testing the final model
I picked Logistic Regression because it's accurate and it also tells me how confident it is. It got about 90% accuracy, so it gets 9 out of 10 reviews right.

In [ ]:
final_model = make_pipeline(LogisticRegression(max_iter=1000, C=2.0))
final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print()
print(classification_report(y_test, y_pred, target_names=["negative", "positive"]))

### Confusion matrix
This chart shows where the model was right and where it made mistakes. Most predictions are on the diagonal, which means they're correct. The mistakes are split evenly between the two classes.

In [ ]:
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["negative", "positive"], yticklabels=["negative", "positive"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Confusion matrix")
plt.show()

### Which words drive the predictions?

### What the model learned
These are the words that push the model's decision the most. Green words make a review look positive and red words make it look negative.

In [ ]:
vocab = final_model.named_steps["tfidf"].get_feature_names_out()
coefs = final_model.named_steps["clf"].coef_[0]
top = pd.concat([
    pd.Series(coefs, index=vocab).nsmallest(15),
    pd.Series(coefs, index=vocab).nlargest(15),
])
top.sort_values().plot.barh(figsize=(7, 8), color=["#d9534f" if c < 0 else "#5cb85c" for c in top.sort_values()])
plt.title("Most negative (red) vs most positive (green) words")
plt.show()

### Try it on your own sentences

I tested the model with sentences I wrote myself. It understood them well, even the tricky ones that use "not".

In [ ]:
samples = [
    "This movie was absolutely fantastic, I loved every minute of it!",
    "Terrible acting and a boring plot. Total waste of time.",
    "It was not good at all.",
    "I didn't expect much but it turned out to be really enjoyable.",
]
for s in samples:
    proba = final_model.predict_proba([clean_text(s)])[0]
    label = "positive" if proba[1] >= 0.5 else "negative"
    print(f"{label:<9} ({max(proba):.0%})  ← {s}")

## Save the model

I saved the trained model to a file called `sentiment_model.pkl` so my Streamlit app can use it without training again.

In [ ]:
joblib.dump(final_model, "sentiment_model.pkl", compress=3)
print("Saved sentiment_model.pkl")

## Run the Streamlit app

I built a simple web app with Streamlit. You type a review and it tells you if it's positive or negative and how confident it is. I deployed it online so anyone can use it.

**Live app:** http://localhost:8501

In [ ]:
import sys
!{sys.executable} -m pip install streamlit
!{sys.executable} -m streamlit run app.py

2026-11-27 16:28:52.277 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.20.10.2:8501

  Help agents write better Streamlit apps?
  Install the official Streamlit skills by running streamlit skills in your terminal.

  For better performance, install the Watchdog module:

  $ xcode-select --install
  $ pip install watchdog
            
